# 01 — I dataset con le etichette MBFC (tentativo 10)

Le nostre testate (scraping del 13 e 18 ottobre) con l'etichetta **factual reporting di MBFC** al posto del livello
NewsGuard: Very Low, Low, Mixed, Mostly Factual, High (Very High, 3 testate, va con High). Come i livelli NewsGuard,
dal peggiore (0) al migliore (4); il **binario** è "almeno Mostly Factual". Fissato il 23 ottobre, prima dei risultati.

Due versioni, con **le stesse testate valutate e gli stessi fold**, così il confronto dice solo l'effetto della pulizia:

- **grezzo**: gli articoli come li ha scaricati lo scraping (`NuovoScraping/dataset_articoli.csv`), tolti solo i testi
  vuoti. Restano anche le copie tra testate di gruppi diversi;
- **pulito**: gli articoli del dataset finale dei tentativi 06–09 (`versioni/dataset_finale.csv`): pulizia della
  Versione 2 (link, tweet, frasi ripetute, copie, articoli prima del 2025) e niente copie tra gruppi diversi.

Le regole sono quelle del tentativo 09:

- **gruppi** = le nostre reti (`02_reti`) ∪ stesso owner NewsGuard ∪ stesso Parent Domain NewsGuard (MBFC non dice
  chi possiede un sito; owner e Parent Domain sono metadati, non etichette);
- **valutate** (il test): almeno 20 articoli nella versione pulita e al più 5 testate per gruppo, scelte a caso con
  seed fisso senza guardare le etichette; le stesse nelle due versioni;
- **di solo training**: le altre testate con un'etichetta MBFC e almeno 5 articoli nella versione, con peso
  min(1, 5/n), n = testate del gruppo;
- **fold**: 5, per gruppo, stratificati per livello MBFC, fatti sulle valutate; le testate di solo training seguono il
  loro gruppo (un gruppo senza valutate: un fold a caso).

Uscite: `Dataset/versioni/<versione>/dataset.csv` e `fold.csv`; in `Results/2-dataset` la composizione.

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

from comune import DA_FACTUAL, DATASET, DATI, LIVELLI, RESULTS, SEED, SOGLIA_BINARIO, VERSIONI, normalizza_owner

MIN_ARTICOLI = 20
MAX_TESTATE_PER_GRUPPO = 5
MIN_ARTICOLI_TRAINING = 5
RESULTS.mkdir(parents=True, exist_ok=True)
mb = pd.read_csv(DATASET / "testate_mbfc.csv").set_index("domain")
mb = mb[mb.factual.notna()]
mb["label"] = mb.factual.map(DA_FACTUAL).astype(int)
print(len(mb), "testate tentate con il factual MBFC")

## Gli articoli delle due versioni

In [ ]:
grezzo = pd.read_csv(DATI / "scraping" / "NuovoScraping" / "dataset_articoli.csv", usecols=["article_id", "dominio", "titolo", "data", "testo"])
grezzo = grezzo.rename(columns={"dominio": "domain", "testo": "text"})
grezzo["domain"] = grezzo.domain.str.lower().str.removeprefix("www.")
grezzo = grezzo[grezzo.text.fillna("").str.strip() != ""]
pulito = pd.read_csv(DATASET / "versioni" / "dataset_finale.csv", usecols=["article_id", "domain", "titolo", "data", "text"])
pulito = pulito[pulito.text.fillna("").str.strip() != ""]
ART = {"grezzo": grezzo[grezzo.domain.isin(mb.index)].reset_index(drop=True), "pulito": pulito[pulito.domain.isin(mb.index)].reset_index(drop=True)}
for v, a in ART.items():
    print(f"{v}: {len(a)} articoli di {a.domain.nunique()} testate con etichetta MBFC")

## I gruppi: reti ∪ owner ∪ Parent Domain (come nel tentativo 09)

In [ ]:
reti = pd.read_csv(DATASET / "derived" / "02_reti.csv").set_index("domain")
ng = pd.read_csv(DATI / "etichette" / "newsguard_2022.csv", low_memory=False)
ng = ng.assign(d=ng.Domain.str.lower()).drop_duplicates("d").set_index("d")
tutte = sorted(set(reti.index) | set(mb.index))
G = nx.Graph(); G.add_nodes_from(tutte)
for _, g in reti.groupby("rete"):
    G.add_edges_from(zip(g.index, g.index[1:]))
for campo, norma in [("Owner", normalizza_owner), ("Parent Domain", lambda x: "" if str(x) == "nan" else str(x).strip().lower())]:
    s = pd.Series({d: norma(ng[campo].get(d, "")) for d in tutte})
    s = s[s != ""]
    for _, g in s.groupby(s):
        G.add_edges_from(zip(g.index, g.index[1:]))
gruppo = {d: i for i, c in enumerate(sorted(nx.connected_components(G), key=len, reverse=True)) for d in c}
mb["gruppo"] = mb.index.map(gruppo); mb["rete"] = mb.index.map(reti.rete)
print(f"{len(set(gruppo.values()))} gruppi; le testate MBFC stanno in {mb.gruppo.nunique()} gruppi")

## Le testate valutate (le stesse nelle due versioni) e i fold

In [ ]:
n_pulito = ART["pulito"].domain.value_counts()
idonee = mb[mb.index.map(n_pulito).fillna(0) >= MIN_ARTICOLI]
valutate = set(idonee.reset_index().sample(frac=1, random_state=SEED).groupby("gruppo").head(MAX_TESTATE_PER_GRUPPO).domain)
val = mb.loc[sorted(valutate)].reset_index()
val["fold_gruppo"] = 0
for i, (_, idx) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(val, val.label, val.gruppo), 1):
    val.loc[idx, "fold_gruppo"] = i
fold_del_gruppo = val.groupby("gruppo").fold_gruppo.first()
rng = np.random.default_rng(SEED)
senza = sorted(set(mb.gruppo) - set(fold_del_gruppo.index))
fold_del_gruppo = pd.concat([fold_del_gruppo, pd.Series(rng.integers(1, 6, len(senza)), index=senza)])
print(f"testate valutate: {len(val)} in {val.gruppo.nunique()} gruppi")
pd.crosstab(val.fold_gruppo, val.label.map(dict(enumerate(LIVELLI))), margins=True, margins_name="tot")

## Le due versioni

In [ ]:
righe = []
for versione, art in ART.items():
    n = art.domain.value_counts()
    art = art[art.domain.map(n) >= MIN_ARTICOLI_TRAINING].copy()
    t = mb.loc[sorted(set(art.domain))][["label", "factual", "factual_score", "credibilita", "orientamento", "gruppo", "rete", "ng_score", "ng_livello"]].copy()
    t["articoli"] = art.domain.value_counts().reindex(t.index)
    t["valutazione"] = t.index.isin(valutate)
    t["testate_nel_gruppo"] = t.groupby("gruppo").label.transform("size")
    t["testate_valutate_nel_gruppo"] = t.groupby("gruppo").valutazione.transform("sum")
    t["peso"] = np.minimum(1.0, MAX_TESTATE_PER_GRUPPO / t.testate_nel_gruppo)
    t["fold_gruppo"] = t.gruppo.map(fold_del_gruppo).astype(int)
    assert t.valutazione.sum() == len(valutate), "una testata valutata manca da questa versione"
    for c in ["label", "gruppo", "rete", "valutazione", "peso", "fold_gruppo"]:
        art[c] = art.domain.map(t[c])
    art["score"] = art.domain.map(t.factual_score)
    (VERSIONI / versione).mkdir(parents=True, exist_ok=True)
    art.to_csv(VERSIONI / versione / "dataset.csv", index=False)
    t.reset_index(names="domain").to_csv(VERSIONI / versione / "fold.csv", index=False)
    for nome, sel in [("valutate", t.valutazione), ("tutte (training)", t.valutazione | True)]:
        L = t.label[sel].map(dict(enumerate(LIVELLI)))
        r = {"versione": versione, "testate": nome, "n": int(sel.sum()), "articoli": int(art.domain.isin(t.index[sel]).sum()),
             "gruppi": int(t.gruppo[sel].nunique()), "affidabili (binario)": float((t.label[sel] >= SOGLIA_BINARIO).mean()),
             "peso totale": float(t.peso[sel].sum())}
        r.update({l: int((L == l).sum()) for l in LIVELLI})
        righe.append(r)
comp = pd.DataFrame(righe).set_index(["versione", "testate"])
comp.to_csv(RESULTS / "01_composizione.csv")
comp.round(3)